# RigTech · runner_colab_continuacao

Notebook de **continuação** na conta Google nova. Roda o pipeline completo a partir do dataset bruto (`Datasets/DaninhasTreinoClientes/`) e cai no LOO (Fase 5).

## Pré-requisitos no Drive novo

- Pasta `MyDrive/Datasets/DaninhasTreinoClientes/` com os 5 talhões (Celso01, CelsoSTE2, DoisRiosFlaviano, Flaviano01, Giasa).
- **Se essa pasta for atalho de 'Compartilhado comigo'**: botão direito → *Organizar → Adicionar atalho ao Drive* → escolher `MyDrive/Datasets/`.

⚠️ **Drive quase cheio** — o backup no fim (célula 8) sobe **só** `history.json`, `weights/*.pt`, `results.csv`, `*.png`. Nunca sobe o dataset materializado nem os tarballs de versão. Se der 'storage full' no rsync, apagar `runs/` antigo no Drive antes de tentar de novo.

Runtime → Change runtime type → **GPU A100** (T4 dobra o tempo).

## 1. Repo e deps

In [ ]:
!git clone https://github.com/maluquintela/rigtech-weed-cycle.git
%cd rigtech-weed-cycle

In [ ]:
!pip install -q ultralytics==8.4.115 rasterio shapely pyproj pillow pyyaml requests

## 2. Drive → disco local

Copiar bruto pra `/content/` — Drive montado é lento demais pra processamento.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Novo caminho: MyDrive/Datasets/DaninhasTreinoClientes/
SRC = '/content/drive/MyDrive/Datasets/DaninhasTreinoClientes'
DRIVE_ROOT = '/content/drive/MyDrive/rigtech-weed-cycle'  # onde vão os artefatos leves
!ls -la "{SRC}" && echo '---' && du -sh "{SRC}"

In [ ]:
!mkdir -p work/geosource/_talhoes
!cp -r "{SRC}"/. work/geosource/_talhoes/
!ls -la work/geosource/_talhoes/

In [ ]:
# Confirmar layout de cada talhão (aceita flat {tif,geojson} OU {imagem/, daninhas/})
!for d in work/geosource/_talhoes/*/; do echo "=== $d ==="; ls "$d"; done

## 3. Fase 1-3 — converter, QA, golden set

In [ ]:
!python -m src.convert_to_yoloseg

In [ ]:
!python -m src.qa_static

In [ ]:
# Hold-out por talhão (Giasa como val, config.yaml). Congela work/golden.
!python -m src.split_golden_by_talhao

In [ ]:
# Sanidade
!echo '--- live/train ---'; ls work/live/images/train 2>/dev/null | wc -l
!echo '--- live/val ---';   ls work/live/images/val   2>/dev/null | wc -l
!echo '--- golden ---';     ls work/golden/images     2>/dev/null | wc -l

## 4. Snapshot v1

In [ ]:
!python -m src.snapshot --note 'v1 colab (conta nova): 5 talhões, Giasa como golden'

## 5. Linear API key (opcional)

Se setada, `train_eval.py` posta o run em Linear no fim. Cole a chave abaixo — vive só na sessão.

In [ ]:
import os
from getpass import getpass
os.environ['LINEAR_API_KEY'] = getpass('LINEAR_API_KEY (Enter pra pular): ') or ''
print('key setada' if os.environ['LINEAR_API_KEY'] else 'sem key — Linear não vai ser atualizado')

## 6. Treino baseline v1 (A100, batch=16, imgsz=1024, amp)

In [ ]:
!python -m src.train_eval --version v1 --tag baseline_colab \
    --device 0 --batch 16 --imgsz 1024 --amp --epochs 100

In [ ]:
!cat work/runs/history.json

## 7. Fase 5 — Leave-one-out por talhão (o bloqueio)

5 treinos (1 por talhão como val). Diagnóstica se o gap de generalização é específico de Giasa ou universal.

**Custo**: ~1.6h/fold × 5 ≈ 8h em T4 (menor em A100). `--skip-existing` deixa retomar se a sessão morrer.

In [ ]:
!git pull

In [ ]:
!python -m src.loo_train \
    --talhoes Celso01 CelsoSTE2 Flaviano01 DoisRiosFlaviano Giasa \
    --device 0 --batch 16 --imgsz 1024 --amp --epochs 100 --skip-existing

## 8. Backup incremental → Drive (só artefatos leves)

Rodar depois de cada fold. ⚠️ **Drive quase cheio** — não sobe dataset materializado nem tarballs.

In [ ]:
!mkdir -p "{DRIVE_ROOT}/runs"
!cp work/runs/history.json "{DRIVE_ROOT}/runs/" 2>/dev/null
!cp work/runs/qa_static.csv "{DRIVE_ROOT}/runs/" 2>/dev/null
!rsync -av --include='*.pt' --include='*.png' --include='*.csv' --include='*.yaml' \
  --include='*/' --exclude='*' work/runs/ "{DRIVE_ROOT}/runs/"

## 9. Ciclo v2 — quando quiser avançar

Suspeitas no golden → Linear → anotador corrige → snapshot v2 → treinar → comparar.

In [ ]:
# 9a. Detectar tiles suspeitas (usa best.pt do último baseline)
!python -m src.suspects

In [ ]:
# 9b. Postar suspeitas em Linear com preview
!python -m src.link_suspects_to_linear --limit 40

In [ ]:
# 9c. Após anotação humana concluída em work/live/, snapshot v2 + treino
# !python -m src.snapshot --note 'v2: correções do ciclo 1 (RIG-XXX..RIG-YYY)'
# !python -m src.train_eval --version v2 --tag pos_ciclo1 --device 0 --batch 16 --imgsz 1024 --amp --epochs 100

In [ ]:
# 9d. Relatório comparativo do ciclo
# !python -m src.cycle_report --from v1 --to v2